# Tutorial 1 — Building a Dataset from Scratch

**Scenario:** You're a computer vision engineer at a wildlife research lab.
You've just received a hard drive with 500 photos from a forest camera trap.
Before you can train a detector, you need to:

1. Organise the images into a managed dataset
2. Record bounding-box annotations for each animal
3. Query the dataset to verify everything looks right
4. Add a second batch of images that arrived a week later

This tutorial walks through each step using the `dman` Python SDK.
We generate synthetic images so you can run it anywhere — no downloads needed.

**Prerequisites:** `pip install dman pillow matplotlib`

## Setup

We isolate everything under a temporary directory so this notebook
never touches your real catalog.  
`DMAN_HOME` must be set **before** importing `dman`.

In [ ]:
import os
import tempfile
import pathlib

# Isolated catalog for this tutorial
tutorial_dir = pathlib.Path(tempfile.mkdtemp(prefix="dman_tutorial1_"))
os.environ["DMAN_HOME"] = str(tutorial_dir / "catalog")

print("Working in:", tutorial_dir)
print("DMAN_HOME: ", os.environ["DMAN_HOME"])

In [ ]:
import subprocess
subprocess.run(["dman", "init"], check=True)

## Step 1 — Simulate the incoming images

In the real scenario these would already exist on disk.
Here we generate 12 tiny colour images with `PIL` — 8 for the initial
delivery and 4 for the "second batch" that arrives later.

In [ ]:
import random
from PIL import Image, ImageDraw, ImageFont

random.seed(42)

# Colour palette: different hues simulate different lighting conditions
BACKGROUNDS = [
    (34, 85, 34),   # forest green
    (139, 115, 85), # muddy brown
    (90, 120, 90),  # dusk grey-green
    (20, 60, 20),   # night
]

images_dir = tutorial_dir / "images"
images_dir.mkdir()

# Each entry: (filename, background_index, animal, bbox as (x,y,w,h) in px)
INITIAL_BATCH = [
    ("trap_001.jpg", 0, "deer",  (20, 30, 60, 80)),
    ("trap_002.jpg", 1, "fox",   (50, 40, 45, 55)),
    ("trap_003.jpg", 0, "deer",  (10, 20, 70, 90)),
    ("trap_004.jpg", 2, "boar",  (30, 50, 80, 60)),
    ("trap_005.jpg", 1, "fox",   (60, 30, 40, 50)),
    ("trap_006.jpg", 3, "deer",  (15, 25, 65, 75)),
    ("trap_007.jpg", 0, "boar",  (40, 45, 75, 65)),
    ("trap_008.jpg", 2, "fox",   (55, 35, 42, 52)),
]

SECOND_BATCH = [
    ("trap_009.jpg", 1, "deer",  (25, 35, 55, 75)),
    ("trap_010.jpg", 3, "boar",  (35, 55, 70, 60)),
    ("trap_011.jpg", 0, "fox",   (45, 25, 48, 58)),
    ("trap_012.jpg", 2, "deer",  (18, 28, 68, 78)),
]

IMG_W, IMG_H = 160, 120

def make_image(path, bg_idx, animal, bbox):
    """Create a synthetic camera-trap image."""
    img = Image.new("RGB", (IMG_W, IMG_H), BACKGROUNDS[bg_idx])
    draw = ImageDraw.Draw(img)
    x, y, w, h = bbox
    # Draw the 'animal' as a filled rectangle with a label
    draw.rectangle([x, y, x + w, y + h], fill=(200, 180, 140), outline=(255, 255, 255))
    draw.text((x + 2, y + 2), animal[:1].upper(), fill=(0, 0, 0))
    img.save(path)

for fname, bg, animal, bbox in INITIAL_BATCH + SECOND_BATCH:
    make_image(images_dir / fname, bg, animal, bbox)

print(f"Generated {len(INITIAL_BATCH) + len(SECOND_BATCH)} images in {images_dir}")

## Step 2 — Create the dataset and add the first batch

`dman.create_dataset()` returns a **builder** — a transaction that
accumulates your changes in memory and commits everything atomically
when you call `.build()`.  This means if something goes wrong halfway
through, the catalog stays clean.

In [ ]:
import dman

builder = dman.create_dataset("wildlife_survey")

# Pre-register the categories we'll use
builder.set_category("deer",  supercategory="ungulate")
builder.set_category("fox",   supercategory="canid")
builder.set_category("boar",  supercategory="suidae")

# Add each image from the initial batch
for fname, _bg, animal, (x, y, w, h) in INITIAL_BATCH:
    img_path = str(images_dir / fname)

    # add_image creates a sample + image asset in one call.
    # The sample name defaults to the filename stem.
    builder.add_image(img_path)

    # Now attach a bounding-box annotation to that sample
    sample_name = pathlib.Path(fname).stem   # e.g. "trap_001"
    builder.add_annotation(
        sample_name,
        category=animal,
        bbox={"x": x, "y": y, "width": w, "height": h},
    )

# Commit — everything above lands in the DB in one transaction
ds = builder.build()

print(f"Dataset created:  {ds.sample_count()} samples, "
      f"{ds.annotation_count()} annotations, "
      f"{ds.category_count()} categories")

## Step 3 — Inspect what's in the dataset

`load_dataset()` returns a **read-only snapshot** of the current state.
It's cheap to call — load it whenever you want a fresh view.

In [ ]:
ds = dman.load_dataset("wildlife_survey")

print(f"{'Sample':<12} {'Asset file':<20} {'Annotation'}")
print("-" * 55)

for sample in ds.samples():
    name = sample["name"]
    asset_file = pathlib.Path(sample["assets"][0]["file_path"]).name
    annots = ds.annotations(name)
    ann_str = ", ".join(a["category_name"] for a in annots) if annots else "—"
    print(f"{name:<12} {asset_file:<20} {ann_str}")

## Step 4 — Visualise a sample

Let's pick a sample and draw its annotation to confirm the bounding box
lines up with the image correctly.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

ANIMAL_COLORS = {"deer": "lime", "fox": "orange", "boar": "red"}

sample = ds.get_sample("trap_004")
img_path = sample["assets"][0]["file_path"]
annots   = ds.annotations("trap_004")

fig, ax = plt.subplots(figsize=(4, 3))
ax.imshow(Image.open(img_path))
ax.set_title("trap_004")

for ann in annots:
    bbox  = json.loads(ann["bbox"])
    color = ANIMAL_COLORS.get(ann["category_name"], "white")
    rect  = mpatches.Rectangle(
        (bbox["x"], bbox["y"]), bbox["width"], bbox["height"],
        linewidth=2, edgecolor=color, facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(bbox["x"], bbox["y"] - 3, ann["category_name"],
            color=color, fontsize=8, fontweight="bold")

ax.axis("off")
plt.tight_layout()
plt.show()

## Step 5 — The second batch arrives

A week later the field team uploads four more images.
`update_dataset()` opens the existing dataset for modification.
Like the builder, changes are held in a transaction until you call `.apply()`.

In [ ]:
updater = dman.update_dataset("wildlife_survey")

for fname, _bg, animal, (x, y, w, h) in SECOND_BATCH:
    img_path = str(images_dir / fname)
    sample_name = pathlib.Path(fname).stem

    # add_sample + add_asset separately — use this when you need full control
    updater.add_sample(sample_name)
    updater.add_asset(
        sample_name,
        asset_type="image",
        file_path=img_path,
        width=IMG_W,
        height=IMG_H,
        metadata=None,
    )

updater.apply()   # commit the second batch

# Reload to see the new state
ds = dman.load_dataset("wildlife_survey")
print(f"After update: {ds.sample_count()} samples")

> **Why are annotations missing for the new batch?**  
> The second batch arrived without labels — that's realistic. You'd
> use a labelling tool (e.g. Label Studio) to annotate them, then call
> `update_dataset()` again to write the annotation results back.

## Step 6 — Summary statistics

Before handing the dataset off to a model trainer, let's do a
quick sanity check: how many images per animal class?

In [ ]:
from collections import Counter

ds = dman.load_dataset("wildlife_survey")

counts = Counter()
for sample in ds.samples():
    for ann in ds.annotations(sample["name"]):
        counts[ann["category_name"]] += 1

print("Annotation counts per class:")
for animal, n in sorted(counts.items()):
    bar = "█" * n
    print(f"  {animal:<6} {bar} ({n})")

unlabelled = sum(1 for s in ds.samples() if not ds.annotations(s["name"]))
print(f"\nSamples without annotations: {unlabelled} "
      f"(the 4 images from the second batch)")

## Cleanup

In [ ]:
import shutil
shutil.rmtree(tutorial_dir)
print("Cleaned up:", tutorial_dir)

---

**What you learned:**

- `create_dataset()` + `.build()` — atomic dataset creation
- `add_image()` for the common case; `add_sample()` + `add_asset()` for full control
- `add_annotation()` with bounding boxes
- `load_dataset()` — read-only snapshot for querying
- `update_dataset()` + `.apply()` — adding data incrementally

**Next:** [Tutorial 2 — Converting Between Formats](tutorial_02_format_conversion.ipynb)